In [10]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"

from jVMC_exp.sharding_config import MESH, DEVICE_SHARDING
import jax

print("Devices =", jax.devices())
print("Device count =", jax.device_count())
print("Mesh shape =", MESH.shape["devices"])

Devices = [CpuDevice(id=0)]
Device count = 1
Mesh shape = 1


In [11]:
import jax
import jax.numpy as jnp
from typing import Callable
from functools import partial

from jVMC_exp.optimizer.base import AbstractOptimizer
from jVMC_exp.objective_function.base import ObjectiveFunctionOutput
from jVMC_exp.sharding_config import DEVICE_SPEC, MESH

class MinSR_old(AbstractOptimizer):
    """
    This class provides functionality for energy minimization via MinSR.

    See `[arXiv:2302.01941] <https://arxiv.org/abs/2302.01941>`_ for details.

    Initializer arguments:
        * ``sampler``: A sampler object.
        * ``pinvTol``: Regularization parameter :math:`\\epsilon_{SVD}`, see above.
        * ``diagonalSchift``: Regularization parameter :math:`\\lambda`, see below.
        * ``diagonalizeOnDevice``: Choose whether to diagonalize :math:`S` on GPU or CPU.
    """
    def __init__(self, sampler, psi, pinvTol=1e-14, diagonalShift=1e-3, centered=True):
        self.pinvTol = pinvTol
        self._diag_shift_fn = diagonalShift if isinstance(diagonalShift, Callable) else lambda step: diagonalShift
        self.diagonalShift = self._diag_shift_fn(0)
        self.holomorphic = psi.holomorphic
        self.real = psi.realParams
        self.centered = centered

        super().__init__(sampler, psi, use_cross_valiadation=False)

    def update_hyperparams(self, step):
        self.diagonalShift = self._diag_shift_fn(step)

    def get_update(self, objective_function_output: ObjectiveFunctionOutput):
        """
        Uses the techique proposed in arXiv:2302.01941 to compute the updates.
        Efficient only if number of samples :math:`\\ll` number of parameters.
        """

        @partial(jax.jit, static_argnames=["padding"])
        @partial(jax.shard_map, mesh=MESH, in_specs=(DEVICE_SPEC, None, None, None, None), out_specs=DEVICE_SPEC)
        def _solve(gradients, eloc, diagonalShift, pinvTol, padding):
            gr = jnp.concatenate([gradients, jnp.zeros((gradients.shape[0], padding))], axis=1)
            gr = jax.lax.all_to_all(gr, 'devices', split_axis=1, concat_axis=0, tiled=True)
            y = gr @ jnp.conj(jnp.transpose(gr)) # (Ns,Ns)
            y = jax.lax.psum(y, 'devices')
            y = y + diagonalShift * jnp.eye(y.shape[-1])
            y = jnp.linalg.pinv(y, rtol=pinvTol, hermitian=True)
            y = y @ eloc # (Ns,)
            y = -1 * jnp.conj(jnp.transpose(gr)) @ y # (Np,)
            return y
        
        gradients_sampobs = objective_function_output.grad_log_psi
        eloc_sampobs = objective_function_output.o_loc
        
        if self.centered:
            gradients_data = gradients_sampobs._centered_obs # (Ns,Np)
            eloc_data = eloc_sampobs._centered_obs.reshape(-1) # (Ns,)
        else:
            gradients_data = gradients_sampobs.observations # (Ns,Np)
            eloc_data = eloc_sampobs.observations.reshape(-1) # (Ns,)

        # padding for all_to_all: the number of parameters needs to be divisible by the number of devices
        a = self.psi.numParameters * (2 if not self.real else 1)
        b = jax.device_count()
        self.missingSize = int((b - a % b) % b)

        if not self.holomorphic and not self.real:
            gradients_data = jnp.concatenate([jnp.real(gradients_data), jnp.imag(gradients_data)], axis=0)
            eloc_data = jnp.concatenate([jnp.real(eloc_data), jnp.imag(eloc_data)], axis=0)
        
        update = _solve(gradients_data, eloc_data, self.diagonalShift, self.pinvTol, self.missingSize)
        update = (update[:-self.missingSize] if self.missingSize > 0 else update)
        update = jnp.array(jax.experimental.multihost_utils.process_allgather(update, tiled=True)).reshape(-1)
        return update
    
    def cross_validation(self):
        raise NotImplementedError
    
    def _update_meta_data(self):
        pass


In [12]:
import jax.numpy as jnp
import jVMC_exp 
jax.config.update("jax_log_compiles", True)

In [4]:
from jVMC_exp.sharding_config import DEVICE_SPEC, REPLICATED_SPEC, REPLICATED_SHARDING, MESH, sharded
from functools import partial

@jax.jit
def _concat_nonholo(arr):
    return jnp.concatenate([jnp.real(arr), jnp.imag(arr)], axis=0)

class Test():
    def __init__(self, diag_shift, pinv_tol):
        self.diag_shift = diag_shift
        self.pinv_tol = pinv_tol
        self._params_pad_size = 0
     
    def get_update(self, psi, objective_function_output):
        """
        Uses the techique proposed in arXiv:2302.01941 to compute the updates.
        Efficient only if number of samples :math:`\\ll` number of parameters.
        """
        grad_log_psi_centered = objective_function_output.grad_log_psi._centered_obs
        o_loc_centered = objective_function_output.o_loc._centered_obs.reshape(-1)  

        if not psi.holomorphic and not psi.realParams:
            grad_log_psi_centered = _concat_nonholo(grad_log_psi_centered)
            o_loc_centered = _concat_nonholo(o_loc_centered)

        update = self._solve(
            grad_log_psi_centered, 
            o_loc_centered, 
            diag_shift=self.diag_shift, 
            pinv_tol=self.pinv_tol, 
            batch_size=None
        )

        # print("aaa", update.shape)

        return jnp.array(jax.experimental.multihost_utils.process_allgather(update, tiled=True))

    @sharded(use_vmap=False, in_specs=(DEVICE_SPEC, REPLICATED_SPEC))
    def _solve(self, gradients, o_loc, *, diag_shift, pinv_tol, batch_size):
        # print(gradients.shape)
        # print(o_loc.shape)
        gradients = jnp.concatenate([
            gradients, 
            jnp.zeros((gradients.shape[0], self._params_pad_size))], axis=1
        )
        # print(gradients.shape)
        gradients = jax.lax.all_to_all(gradients, 'devices', split_axis=1, concat_axis=0, tiled=True)
        # print(gradients.shape)
        y = gradients @ jnp.conj(jnp.transpose(gradients))                          # (Ns, Ns)
        # print(y.shape)
        y = jax.lax.psum(y, 'devices')
        # print(y.shape)
        y = y + diag_shift * jnp.eye(y.shape[-1])
        # print(y.shape)

        y = jnp.linalg.pinv(y, rtol=pinv_tol, hermitian=True)
        # print(y.shape)
        y = y @ o_loc                                               # (Ns,)
        # print(y.shape)
        y = -1 * jnp.conj(jnp.transpose(gradients)) @ y             # (Np,)
        # print(y.shape)

        out = y[:-self._params_pad_size] if self._params_pad_size > 0 else y
        # print("out", out.shape)

        return out
    
test_cls = Test(1e-3, 1e-8)

In [13]:
L = 4
num_samples = 2**L
num_chains = 2**3
batch_size = num_samples

net = jVMC_exp.nets.CpxRBM(3, bias=True)
psi = jVMC_exp.vqs.NQS(net, L, batch_size, seed=123)
proposer = jVMC_exp.propose.SpinFlip()
# sampler = jVMC_exp.sampler.MCSampler(psi, proposer, 123, num_chains, num_samples)
sampler = jVMC_exp.sampler.ExactSampler(psi)
loss_function = jVMC_exp.objective_function.Observable(jVMC_exp.operator.discrete.SigmaX(0))
opt = jVMC_exp.optimizer.MinSR(sampler, psi)
opt_old = MinSR_old(sampler, psi)

Finished tracing <lambda> for jit in 0.000283718 sec
Finished tracing <lambda> for jit in 0.000291348 sec
Finished tracing <lambda> for jit in 0.000250101 sec
Finished tracing <lambda> for jit in 0.000188351 sec
Finished tracing <lambda> for jit in 0.001275063 sec
Finished tracing <lambda> for jit in 0.000331879 sec


In [6]:
psi.numParameters, psi.parameters_flat.shape

(Array(15, dtype=int64), (30,))

In [15]:
loss_function_out = loss_function.value_and_grad(sampler)

In [8]:
test_cls.get_update(psi, loss_function_out).shape

(1, 30)

In [17]:
opt.get_update(loss_function_out).shape

(30,)

In [11]:
psi.numParameters, psi.parameters_flat.shape

(Array(15, dtype=int64), (30,))

In [94]:
opt.get_update(loss_function_out).shape

(16,)

In [95]:
import unittest
import tqdm
import jax.numpy as jnp

import jVMC_exp
import jVMC_exp.nets as nets
from jVMC_exp.vqs import NQS
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler

L = 4

batch_size = int(2 ** L)
# Set up variational wave function
rbm = nets.CpxRBM(numHidden=3, bias=False)
psi = NQS(rbm, L, batch_size, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
loss_function = jVMC_exp.objective_function.Observable(jVMC_exp.operator.discrete.SigmaX(0))
stepper = jVMC_exp.stepper.Euler(1e-2)
opt = jVMC_exp.optimizer.MinSR(exact_sampler, psi, pinv_tol=1e-6, diagonalShift=1e-3)
# opt = jVMC_exp.optimizer.SR(exact_sampler, psi)

In [96]:
psi.numParameters, psi.parameters_flat.shape

(Array(12, dtype=int64), (24,))

In [76]:
opt.get_update(loss_function.value_and_grad(exact_sampler)).shape

(16,)

In [59]:
opt.step(0, jVMC_exp.stepper.Euler(1), loss_function)[0].shape

TypeError: add got incompatible shapes for broadcasting: (24,), (16,).